## Global Imports

In [ ]:
# Helper function to check if the code is running in Google Colab
def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

# Usage
if is_colab():
    print("Running in Google Colab")
else:
    print("Not running in Google Colab")


# Importing libraries
import pandas as pd
import numpy as np
from scipy.io import loadmat
from IPython.display import display
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast  # Add these imports for mixed precision training
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, TensorDataset
from torchsummary import summary
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, auc
import sklearn.preprocessing
from tqdm.auto import tqdm
import seaborn
from enum import Enum


import os
import shutil
import subprocess
import zipfile
import re
import random
import h5py




## Global Definitions

In [ ]:
# Path to the zip file
data_dir = "./Data"
google_drive_folder = "/content/drive/MyDrive/UU"
zip_path = f"{data_dir}/Final Project data.zip"

# Set the device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set pandas display options to show all columns
pd.set_option('display.max_columns', None)

# If we are in colab and we need to mount the drive
if is_colab():
  from google.colab import drive
  # Mount Google Drive
  drive.mount('/content/drive')
  print("Google Drive mounted successfully.")
  zip_path = f"{google_drive_folder}/Final Project data.zip"
else:
  print("Not in Google Colab.")

In [ ]:
# Check if CUDA is available
print(torch.__version__)
print(torch.version.cuda)  # Should match CUDA 12.5
print(torch.backends.cudnn.version())  # Should return a version, not None
print(torch.cuda.is_available())  # Should return True

## Extract the Zip File

In [ ]:
# Check if the dataset is already extracted, if not extract it
if os.path.exists(data_dir):
    print("Extracting the dataset...")
    #unzip
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    #list files in the extracted dir
    os.listdir(data_dir)
else:
    print("Dataset already extracted.")

## Utils

In [ ]:
import h5py

def get_dataset_from_h5py(filename_path):
    """
        Get the dataset from the h5py file.
        Parameters:
        filename_path (str): The path to the h5py file.
        Returns:
        matrix (numpy.ndarray): The dataset as a numpy array.
    """

    matrix = None
    try:
        with h5py.File(filename_path, 'r') as f:
            # List all datasets or groups in the file
            avilable_datasets = list(f.keys())
            if len(avilable_datasets) == 0:
                print("No datasets or groups found in the file.")
            else:
                print(f"Available datasets file {filename_path}: {avilable_datasets}")

            dataset_name = avilable_datasets[0]
            # Check if the dataset exists in the file
            if dataset_name in f:
                matrix = f.get(dataset_name)[()]
            else:
                print(f"Dataset '{dataset_name}' not found in the file.")
    except OSError as e:
        print(f"Error opening file {filename_path}: {e}")
    except Exception as e:
        print(f"An error occurred: {e}")
    matrix = scale_data(matrix, NormalizationType.MINMAX_LOCAL)
    return torch.tensor(matrix)


## Load Data

In [ ]:
train_data = []
tasks = ["rest", "task_motor", "task_story_math", "task_working_memory"]
for task in tasks:
    for trial in range(1,9):
        dataset = get_dataset_from_h5py(f"{data_dir}/Final Project data/Intra/train/{task}_105923_{trial}.h5")
        
        # Append this trial to the list
        train_data.append(dataset)

# Stack in 3rd dimension
train_data = torch.stack(train_data)

print(train_data.shape)

In [ ]:
test_data = []
tasks = ["rest", "task_motor", "task_story_math", "task_working_memory"]
for task in tasks:
    for trial in range(9, 10):
        dataset = get_dataset_from_h5py(f"{data_dir}/Final Project data/Intra/test/{task}_105923_{trial}.h5")
        
        # Append this trial to the list
        test_data.append(dataset)

# Stack in 3rd dimension
test_data = torch.stack(test_data)

print(test_data.shape)

In [ ]:
class NormalizationType(Enum):
    MINMAX_LOCAL = 1
    Z_SCORE_LOCAL = 2
    Z_SCORE_TIMEWISE = 3


def scale_data(data, scaler_type):
    if(scaler_type == NormalizationType.MINMAX_LOCAL):
        scaler = sklearn.preprocessing.MinMaxScaler()
        return(scaler.fit_transform(data))
    elif(scaler_type == NormalizationType.Z_SCORE_LOCAL):
        scaler = sklearn.preprocessing.StandardScaler()
        return(scaler.fit_transform(data))
    elif(scaler_type == NormalizationType.Z_SCORE_TIMEWISE):
        mean = torch.mean(train_data, 0)
        sd = torch.std(train_data, 0)
        return((data-mean)/sd)

## Model

In [ ]:
class MEGClassifier(nn.Module):
    def __init__(self, input_channels=248, seq_length=35624, num_classes=4):
        super(MEGClassifier, self).__init__()

        # 1D CNN for Temporal Feature Extraction
        self.cnn1d = nn.Sequential(
            nn.Conv1d(input_channels, 128, kernel_size=7, stride=1, padding=3),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4)
        )

        # 2D CNN for Spatial Feature Extraction
        self.cnn2d = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=(3, 3), padding=(1, 1)),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)),

            nn.Conv2d(64, 128, kernel_size=(3, 3), padding=(1, 1)),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)),

            nn.Conv2d(128, 256, kernel_size=(3, 3), padding=(1, 1)),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2))
        )

        # Transformer Encoder for Temporal Attention
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=256, nhead=8, dim_feedforward=512),
            num_layers=4
        )

        # Fully Connected Classification Layer
        self.fc = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # 1D CNN
        x = self.cnn1d(x)

        # Reshape for 2D CNN (batch, 1, channels, seq_len)
        x = x.unsqueeze(1)
        x = self.cnn2d(x)

        # Reshape for Transformer (batch, seq_len, feature)
        x = x.flatten(2).permute(2, 0, 1)  # (seq_len, batch, 256)
        x = self.transformer(x)  # (seq_len, batch, 256)

        # Global Average Pooling
        x = x.mean(dim=0)  # (batch, 256)

        # Classification
        x = self.fc(x)

        return x

# Model instantiation
model = MEGClassifier()

# Example input: Batch of 8 samples
dummy_input = torch.randn(8, 248, 35624)
output = model(dummy_input)
print("Output shape:", output.shape)  # Expected: (8, 4)
print(output)

In [ ]:
def train_meg_classifier(model, train_loader, val_loader, num_epochs=25, lr=0.001, device='cpu'):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        model.train()
        train_loss, correct, total = 0, 0, 0

        for data, labels in train_loader:
            data, labels = data.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * data.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_acc = 100.0 * correct / total
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {train_loss / total:.4f}, Accuracy: {train_acc:.2f}%")

        # Validation
        model.eval()
        val_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for data, labels in val_loader:
                data, labels = data.to(device), labels.to(device)
                outputs = model(data)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * data.size(0)

                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        val_acc = 100.0 * correct / total
        print(f"Validation Loss: {val_loss / total:.4f}, Validation Accuracy: {val_acc:.2f}%")


In [ ]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(test_data, batch_size=32, shuffle=False)
model = MEGClassifier()
train_meg_classifier(model, train_loader, val_loader)
